In [1]:
import pandas as pd
import numpy as np
import os
from data_preprocessor import DataPreprocessor
from state_analyzer import SystemStateAnalyzer
from state_clusterer import StateClusterer
from state_visualizer import StateVisualizer

class Config:
    def __init__(self):
        self.data_path = "data/processed_robots_data.parquet"
        self.output_dir = "results/state_analysis"
        self.dt = 0.1
        self.n_clusters = 4
        self.random_state = 42

def ensure_dir(directory):
    if not os.path.exists(directory):
        os.makedirs(directory)

def main():
    config = Config()
    ensure_dir(config.output_dir)
    
    print("1. Загрузка и предобработка данных...")
    preprocessor = DataPreprocessor(config.data_path, config.dt)
    experiments_dict = preprocessor.load_and_vectorize()
    
    print("2. Вычисление кинематики...")
    kinematics_dict = preprocessor.compute_kinematics_vectorized(experiments_dict)
    
    print("3. Извлечение признаков состояний...")
    analyzer = SystemStateAnalyzer(dt=config.dt)
    features_dict = analyzer.extract_features_batch(experiments_dict, kinematics_dict)
    
    print("4. Кластеризация K-means...")
    clusterer = StateClusterer(n_clusters=config.n_clusters, random_state=config.random_state)
    features_array = clusterer.prepare_features(features_dict)
    cluster_labels, umap_embedding, X_scaled = clusterer.fit_cluster(features_array)
    
    print("5. Анализ кластеров...")
    cluster_stats = clusterer.analyze_clusters(features_array, cluster_labels)
    
    print("6. Визуализация...")
    clusterer.visualize_clusters(umap_embedding, cluster_labels, 
                                save_path=f'{config.output_dir}/clusters_umap.png')
    
    results_df = clusterer.save_results(features_array, cluster_labels, umap_embedding,
                                       output_path=f'{config.output_dir}/cluster_results.csv')
    
    visualizer = StateVisualizer()
    visualizer.plot_cluster_stats(cluster_stats, 
                                 save_path=f'{config.output_dir}/cluster_statistics.png')
    visualizer.plot_feature_correlation(results_df, 
                                       save_path=f'{config.output_dir}/feature_correlation.png')
    visualizer.plot_cluster_distribution(results_df, 
                                        save_path=f'{config.output_dir}/cluster_distribution.png')
    
    interpretation_df = visualizer.create_state_interpretation_table(cluster_stats)
    interpretation_df.to_csv(f'{config.output_dir}/state_interpretation.csv', index=False)
    
    print("\n7. Результаты:")
    print(f"   Всего образцов: {len(features_array)}")
    print(f"   Количество кластеров: {config.n_clusters}")
    print(f"\nИнтерпретация кластеров:")
    print(interpretation_df.to_string(index=False))
    
    print(f"\nРезультаты сохранены в: {config.output_dir}/")
    print(f"  - cluster_results.csv (все данные)")
    print(f"  - state_interpretation.csv (интерпретация)")
    print(f"  - *.png (визуализации)")

if __name__ == "__main__":
    main()

1. Загрузка и предобработка данных...
📥 Векторизованная загрузка данных...
🎯 Загружено 307 экспериментов на cuda
2. Вычисление кинематики...
🔬 Векторизованное вычисление кинематики...
3. Извлечение признаков состояний...
4. Кластеризация K-means...
5. Анализ кластеров...
6. Визуализация...

7. Результаты:
   Всего образцов: 181894
   Количество кластеров: 4

Интерпретация кластеров:
 Cluster  Samples                        State_Type    PO Angular_Vel Coord_Num Mean_Dist Vel_Disp
       0    21332       Regular Structure (Мицелла) 0.137       0.037       4.5     361.0    0.706
       1    85577       Regular Structure (Мицелла) 0.134       0.036       4.1     377.4    0.678
       2    11820             Chaotic Motion (Хаос) 0.133       0.062       3.6     394.6    0.819
       3    63165 Sparse Distribution (Разреженное) 0.133       0.031       3.5     420.1    0.674

Результаты сохранены в: results/state_analysis/
  - cluster_results.csv (все данные)
  - state_interpretation.csv (инт